In [ ]:
"""
==============================================================================
🚀 [Step 2] Data Stratification & Copy-Paste Augmentation
==============================================================================
### @Author       : 한의정 (Data Engineering Lead)
### @Description  : 계층적 분할(Stratified Split)을 통한 독립적 Val 셋 구축 및
###                 희귀 클래스 물량 확보를 위한 GrabCut 마스크 기반 Copy-Paste 수행.
### @Input        : merged_annotations_train_final.json, train_images/
### @Output       : train_raw.json, val.json, crops_minority/, train_augmented_final.json
==============================================================================
"""


In [ ]:
import sys, os

try:
    is_colab = 'google.colab' in str(get_ipython())
except NameError:
    is_colab = False

if is_colab:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass  # 이미 마운트됐거나 VSCode 커널 환경

    REPO_DIR = '/content/pill_detection_project'
    if not os.path.exists(REPO_DIR):
        os.system('git clone https://github.com/wina0901/pill_detection_project.git ' + REPO_DIR)

    # nested 폴더 대응 (pill_detection_project/pill_detection_project/src/ 구조)
    PROJECT_ROOT = REPO_DIR
    nested = os.path.join(REPO_DIR, 'pill_detection_project')
    if os.path.isdir(os.path.join(nested, 'src')):
        PROJECT_ROOT = nested

    sys.path.insert(0, PROJECT_ROOT)
    BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'

else:
    REPO_DIR = os.path.expanduser('~/Desktop/vera_pill')
    sys.path.insert(0, REPO_DIR)
    BASE_DIR = os.path.expanduser('~/Desktop/vera_pill/dataset')

print(f"환경    : {'Colab' if is_colab else '로컬'}")
print(f"REPO_DIR: {sys.path[0]}")
print(f"BASE_DIR: {BASE_DIR}")
assert os.path.exists(BASE_DIR), f"🚨 경로 없음: {BASE_DIR}"


In [ ]:
import json, os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split
from src.preprocessing.augmentation import extract_minority_crops, run_copy_paste, make_pill_mask
from src.preprocessing.viz_utils import (show_samples, show_augmented_samples,
                                          show_mask_preview, show_class_distribution,
                                          show_aug_vs_original)

plt.rcParams['font.family']        = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')
print("✅ 라이브러리 로드 완료")


## 1. 원본 JSON 경로 확인

In [ ]:
original_json_path  = os.path.join(BASE_DIR, 'merged_annotations_train_final.json')
train_raw_json_path = os.path.join(BASE_DIR, 'train_raw.json')
val_json_path       = os.path.join(BASE_DIR, 'val.json')

assert os.path.exists(original_json_path), f"JSON 없음: {original_json_path}"
assert os.path.exists(os.path.join(BASE_DIR, 'train_images')), "train_images/ 폴더 없음"

print("🚀 [Step 1] 원본 통합 COCO 데이터 로딩 중...")
with open(original_json_path, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)

images     = coco_data['images']
annotations = coco_data['annotations']
categories  = coco_data['categories']

print(f"이미지: {len(images):,}장 / 어노테이션: {len(annotations):,}개 / 클래스: {len(categories)}종")


## 2. Stratified Split (train / val 분리)
> 73 클래스 불균형 때문에 랜덤 split이 아닌 계층화 split을 씁니다.

In [ ]:
# ==========================================
# 1. 이미지별 대표 클래스(Dominant Class) 추출
# ==========================================
# Multi-label 이미지(알약 여러 개)를 단일 Label 기준으로 스플릿하기 위한 휴리스틱
# 한 이미지에 여러 클래스가 있을 때 가장 많이 등장한 클래스를 '대표 클래스'로 선정

img_to_cats = defaultdict(list)
for ann in annotations:
    img_to_cats[ann['image_id']].append(ann['category_id'])

def get_majority_label(cat_list):
    """이미지 내 최빈 클래스를 해당 이미지의 대표 클래스로 선정합니다."""
    return max(set(cat_list), key=cat_list.count)

img_dominant = {
    img_id: get_majority_label(cats)
    for img_id, cats in img_to_cats.items()
}

# ==========================================
# 2. 계층적 분할 (Stratified Split Engine)
# ==========================================
print("⚖️ [Step 2] 클래스 비율을 보존하며 Train/Val 9:1 분할을 시작합니다...")

# 재현성(Reproducibility)을 위한 시드 고정
random.seed(42)

class_to_imgs = defaultdict(list)
for img_id, label in img_dominant.items():
    class_to_imgs[label].append(img_id)

train_ids, val_ids = set(), set()

for label, img_list in class_to_imgs.items():
    random.shuffle(img_list)  # 시드 고정으로 매번 동일하게 섞임

    if len(img_list) == 1:
        # [예외 1] 극소수 클래스 (1장): 무조건 Train
        # → Validation에서 mAP 0이 되는 것보다 최소 1번이라도 학습시키는 게 낫습니다
        train_ids.update(img_list)
    elif len(img_list) < 5:
        # [예외 2] 소수 클래스 (5장 미만): Val에 최소 1장 보장
        # → 소수 클래스도 평가 가능하도록 Val에 1장은 남겨둡니다
        val_ids.add(img_list[0])
        train_ids.update(img_list[1:])
    else:
        # [일반] 충분한 데이터: 정확히 9:1 비율로 스플릿
        split_idx = max(1, int(len(img_list) * 0.1))
        val_ids.update(img_list[:split_idx])
        train_ids.update(img_list[split_idx:])

print(f"📊 [결과] 총 {len(images):,}장 → Train: {len(train_ids):,}장 / Val: {len(val_ids):,}장")

# ==========================================
# 3. 분할 무결성(Integrity) 검증
# ==========================================
val_classes = set(img_dominant[i] for i in val_ids if i in img_dominant)
missing     = set(img_dominant.values()) - val_classes

if missing:
    print(f"🚨 [경고] Val 셋에 누락된 클래스 {len(missing)}개 (사유: 전체 데이터 1장뿐인 클래스)")
else:
    print(f"✅ [검증] 전체 {len(categories)}개 클래스가 Val 셋에 모두 포함되었습니다.")


In [ ]:
# ==========================================
# 4. JSON 분리 + 저장
# ==========================================
# O(1) 탐색을 위해 Set 자료형 활용 (속도 극대화)
train_images      = [img for img in images     if img['id']       in train_ids]
val_images        = [img for img in images     if img['id']       in val_ids]
train_annotations = [ann for ann in annotations if ann['image_id'] in train_ids]
val_annotations   = [ann for ann in annotations if ann['image_id'] in val_ids]

train_coco = {"images": train_images, "annotations": train_annotations, "categories": categories}
val_coco   = {"images": val_images,   "annotations": val_annotations,   "categories": categories}

# indent=2를 빼면 JSON 용량이 절반 이하로 줄고 로딩 속도가 3배 이상 빨라집니다.
# 사람이 직접 읽을 목적이 아니라면 indent 제거를 권장합니다.
with open(train_raw_json_path, 'w', encoding='utf-8') as f:
    json.dump(train_coco, f, ensure_ascii=False, indent=2)
with open(val_json_path, 'w', encoding='utf-8') as f:
    json.dump(val_coco, f, ensure_ascii=False, indent=2)

print(f"✅ train_raw.json 저장: {len(train_images)}장 / {len(train_annotations):,}개")
print(f"✅ val.json 저장      : {len(val_images)}장 / {len(val_annotations):,}개")


## 3. 클래스 분포 확인 (Split 결과)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

for ax, json_p, title in [
    (axes[0], train_raw_json_path, "Train 클래스 분포"),
    (axes[1], val_json_path,       "Val 클래스 분포"),
]:
    with open(json_p) as f:
        d = json.load(f)
    cat_map = {c['id']: c['name'] for c in d['categories']}
    cnts    = defaultdict(int)
    for ann in d['annotations']:
        cnts[cat_map[ann['category_id']]] += 1
    vals = sorted(cnts.values(), reverse=True)
    ax.bar(range(len(vals)), vals, color='steelblue', width=0.8)
    ax.axhline(50, color='red', linestyle='--', alpha=0.6, label='소수 기준(50)')
    ax.set_title(title); ax.set_xlabel('클래스 (정렬됨)'); ax.set_ylabel('객체 수')
    ax.legend()

plt.tight_layout(); plt.show()


## 4. 소수 클래스 스티커 추출

In [ ]:
extract_minority_crops(BASE_DIR, threshold=50)


## 5. 📸 마스크 추출 결과 확인 (10개)
> 알약 모양대로 누끼가 잘 따졌는지 확인. 빨간 avg = 스킵 대상

In [ ]:
CROP_DIR = os.path.join(BASE_DIR, 'crops_minority')
show_mask_preview(CROP_DIR, n=10)


## 6. Copy-Paste 증강 실행

In [ ]:
run_copy_paste(BASE_DIR, aug_count=1500, random_seed=42)


## 7. 📸 증강 결과 20장 확인
> 사각형 경계선 없이 자연스럽게 합성됐는지 반드시 육안 확인!

In [ ]:
AUG_IMG_DIR = os.path.join(BASE_DIR, 'train_augmented_images')
AUG_JSON    = os.path.join(BASE_DIR, 'train_augmented_final.json')
show_augmented_samples(AUG_IMG_DIR, AUG_JSON, n=20)


In [ ]:
import json
from collections import Counter

with open('/content/drive/MyDrive/data/초급_프로젝트/dataset/merged_annotations_train_final_backup.json') as f:
    orig = json.load(f)
with open('/content/drive/MyDrive/data/초급_프로젝트/dataset/merged_annotations_train_final.json') as f:
    new = json.load(f)

cat_map = {c['id']: c['name'] for c in new['categories']}
orig_counts = Counter(a['category_id'] for a in orig['annotations'])
new_counts  = Counter(a['category_id'] for a in new['annotations'])

print(f"backup: 이미지 {len(orig['images'])}장")
print(f"현재:   이미지 {len(new['images'])}장")
print()
print(f"{'클래스명':<40} {'원본':>6} {'현재':>6} {'추가':>6}")
print('-' * 60)

for cat_id in sorted(new_counts.keys()):
    o = orig_counts.get(cat_id, 0)
    n = new_counts.get(cat_id, 0)
    if n - o > 0:
        name = cat_map.get(cat_id, str(cat_id))
        print(f"{name:<40} {o:>6} {n:>6} {n-o:>+6}")

# [ADR] 데이터셋 어노테이션 누락 이슈 및 증강 파이프라인 진행 결정

| | |
|---|---|
| **Date** | 2026-03-24 |
| **Author** | Data Engineering Team |
| **Status** | `결정 완료 (Approved)` |
| **Context** | Copy-Paste 데이터 증강(v4) 과정 중 원본 데이터의 라벨링 결함 처리 방안 |

---

## 1. 이슈 배경 (Background)

`run_copy_paste` 파이프라인 실행 중 **합성된 알약이 원본 이미지의 기존 알약과 겹치는 현상** 발견.
원인은 `check_overlap` 버그가 아닌, 원본 `train_raw.json` 내 **BBox 어노테이션이 누락된 이미지(False Negative)** 가 섞여 있어 시스템이 해당 영역을 빈 배경으로 오인한 것.

---

## 2. 대안 및 위험성 평가 (Alternatives & Risk Assessment)

| 대안 | 내용 | 결정 | 사유 |
|---|---|---|---|
| **A** | 어노테이션 없는 이미지를 배경에서 제외 | 기각 | 부분 누락 이미지는 여전히 선택됨. 모델에게 Focal Loss 상승 및 Recall 저하 유발 |
| **B** | 전수 재라벨링 또는 AI 자동 검수 | 기각 | 빠른 이터레이션이 중요한 시점에서 라벨 클렌징은 ROI 비효율 |
| **C** | 현 상태 유지 및 증강 강행 | **채택** | 라벨 노이즈(1~2%) 감수, v4 파이프라인 그대로 가동 |

---

## 3. 의사결정 근거 (Rationale)

| # | 근거 |
|---|---|
| 1 | **모델 노이즈 강건성** — YOLO·RetinaNet은 극소수 False Negative를 Smoothing할 충분한 맷집을 가짐 |
| 2 | **원인 분리** — 라벨 누락은 원본 데이터 수집 단계의 한계이며 GrabCut v4 파이프라인의 결함이 아님 |
| 3 | **기대 효과 우위** — 소수 클래스 증강 500장 투입으로 얻는 mAP 상승효과가 압도적으로 큼 |

---

## 4. 파이프라인 핵심 기술 요약 (v4 Engine)

| 기술 | 내용 |
|---|---|
| **정밀 마스킹** | GrabCut 알고리즘으로 HSV 방식의 한계(보호색, C자 구멍) 극복 |
| **내부 노이즈 제거** | `cv2.connectedComponents` + Morphology Close로 알약 내부 음각/그림자 구멍 복원 |
| **자연스러운 엣지** | 동적 Gaussian Blur 커널로 Feathering 구현 (사각형 아티팩트 제거) |

---

## 5. Next Action Items

| 상태 | 항목 |
|---|---|
| ✅ | 증강 파이프라인(v4) 의사결정 합의 완료 |
| ✅ | `run_copy_paste()` 실행 → 소수 클래스 500장 합성 완료 |
| ✅ | Letterbox + CLAHE 전처리 완료 |
| ⬜ | 모델 학습 (Faster R-CNN / RetinaNet / YOLO) 런칭 |
| ⬜ | 캐글 제출 및 리더보드 mAP 갱신 확인 |


## ✅ NB02 완료
> 다음 단계: NB03 - Letterbox 변환